# Mini Project 1 — Analysis Notebook

**Your name:** Gena Lee

**Dataset:**  Last.fm_data.json

**Date:**  5/6/2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [22]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** My data comes from the Last.fm API (last.fm/api), a free public API that aggregates real listening behavior from millions of users across streaming platforms like Spotify and Apple Music. Last.fm tracks individual song plays and exposes aggregated statistics through its API.

I will access the data using Python's requests library, pulling JSON responses directly into pandas DataFrames. No authentication beyond a free API key is required for the public read-only endpoints I plan to use.

**Why this dataset:** *I chose the Last.fm API because of my interests in music and how it relates on a global scale. I think it would be super interesting to see how music is perceived and listened to across countries. I would also be open to focusing on specific countries (e.g. US, Korea, Brazil) if I have time.*

**Three analytical questions:**

1. *Who are the top Last.fm artists by profile country (US, Brazil, South Korea) and how much do they overlap?*
2. *Which genres have the most listeners?*
3. *Among the top 100 global artists, which genres have the highest play-to-listener ratio?*


**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [23]:
import json
from pathlib import Path

#I made a json file to store the data from the last.gm API
with open("Last.fm_Data.json", "r", encoding="utf-8") as f:
    root = json.load(f)

print("Top-level keys:")
for k, v in root.items():
    if isinstance(v, dict) and "records" in v:
        print(f"  {k:25s}  ({len(v['records'])} rows, columns={v['columns']})")
    else:
        print(f"  {k:25s}  ({type(v).__name__})")

Top-level keys:
  _snapshot_meta             (dict)
  df                         (60 rows, columns=['country_label', 'country_param', 'name', 'listeners', 'url', 'streamable', 'image', '@attr.rank', 'mbid'])
  names_by_country           (10 rows, columns=['rank', 'Brazil', 'South Korea', 'United States'])
  pairwise_df                (3 rows, columns=['profile_country_a', 'profile_country_b', 'overlap_artists', 'jaccard_similarity'])
  unique_df                  (3 rows, columns=['profile_country', 'artists_only_on_this_chart', 'examples'])
  rq1_overlap_summary        (1 rows, columns=['key', 'count', 'names_csv'])
  tags_df                    (50 rows, columns=['name', 'url', 'reach', 'taggings', 'streamable'])
  chart100_df                (99 rows, columns=['name', 'playcount', 'listeners', 'mbid', 'url', 'streamable', 'image'])
  rq3_artists_df             (98 rows, columns=['name', 'mbid', 'playcount', 'listeners', 'play_per_listener', 'genre_tag'])
  rq3_genre_summary          (5

In [24]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   country_label  60 non-null     str   
 1   country_param  60 non-null     str   
 2   name           60 non-null     str   
 3   listeners      60 non-null     int64 
 4   url            60 non-null     str   
 5   streamable     60 non-null     str   
 6   image          60 non-null     object
 7   rank           60 non-null     int64 
 8   mbid           58 non-null     str   
dtypes: int64(2), object(1), str(6)
memory usage: 4.3+ KB


In [25]:
# Summary statistics for numeric columns
df.describe()

,listeners,rank
count,60.000000,60.000000
mean,94213.133333,10.500000
std,80549.823316,5.814943
min,496.000000,1.000000
25%,633.250000,5.750000
50%,89887.000000,10.500000
75%,166472.250000,15.250000
max,242926.000000,20.000000


**Your data profile notes:**  
*I found that the original ranking of the top artists weren't ranked by playcount or listeners. So, I had to write a new code to rank the artists by playcount, which gave me a more accurate ranking.*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Research Question 1:** *Who are the top Last.fm artists by profile country (US, Brazil, South Korea) and how much do they overlap?*

In [26]:
# Your analysis for Question 1
import pandas as pd

#root is the json file, which I labeled in the previous cell
#I am relabeling the df section as block, which has a list of columns and records
block = root["df"]
#I am creating a dataframe from the records and columns in the block
df = pd.DataFrame(block["records"], columns=block["columns"])

#I am printing the shape and head of the dataframe
print(df.shape)
df.head()

(60, 9)


,country_label,country_param,name,listeners,url,streamable,image,@attr.rank,mbid
0,United States,United States,Kanye West,242926,https://www.last.fm/music/Kanye+West,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,1,NaN
1,United States,United States,Drake,238213,https://www.last.fm/music/Drake,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,2,9fff2f8a-21e6-47de-a2b8-7f449929d43f
2,United States,United States,Kendrick Lamar,228488,https://www.last.fm/music/Kendrick+Lamar,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,3,381086ea-f511-4aba-bdf9-71c753dc5077
3,United States,United States,"Tyler, The Creator",228055,"https://www.last.fm/music/Tyler,+The+Creator",0,[{'#text': 'https://lastfm.freetls.fastly.net/...,4,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6
4,United States,United States,PinkPantheress,214472,https://www.last.fm/music/PinkPantheress,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,5,7441014f-f8f5-494f-81db-ff166fbc078d


In [27]:
# Clean the RQ1 dataframe

# Rename the Last.fm rank column to something easier to work with
df = df.rename(columns={"@attr.rank": "rank"})

# Convert rank and listeners from text to numbers
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
df["listeners"] = pd.to_numeric(df["listeners"], errors="coerce")

# Keep only the columns needed for Research Question 1
rq1_df = df[["country_label", "name", "rank", "listeners"]].copy()

print(rq1_df.shape)
rq1_df.head()

(60, 4)


,country_label,name,rank,listeners
0,United States,Kanye West,1,242926
1,United States,Drake,2,238213
2,United States,Kendrick Lamar,3,228488
3,United States,"Tyler, The Creator",4,228055
4,United States,PinkPantheress,5,214472


In [28]:
top_10_by_country = rq1_df[rq1_df["rank"] <= 10].sort_values(
    ["country_label", "rank"]
)

top_10_by_country

,country_label,name,rank,listeners
20,Brazil,Lady Gaga,1,117912
21,Brazil,Sabrina Carpenter,2,107347
22,Brazil,Ariana Grande,3,104454
23,Brazil,Taylor Swift,4,103022
24,Brazil,Marina Sena,5,100823
25,Brazil,Anitta,6,98481
26,Brazil,Lana Del Rey,7,97000
27,Brazil,Michael Jackson,8,96004
28,Brazil,The Weeknd,9,94252
29,Brazil,Rihanna,10,90366


In [38]:
#I am showing the overlapping data of the three countries

pairwise_block = root["pairwise_df"]
pairwise_df = pd.DataFrame(
    pairwise_block["records"],
    columns=pairwise_block["columns"]
)

pairwise_df

,profile_country_a,profile_country_b,overlap_artists,jaccard_similarity
0,Brazil,South Korea,10,0.3333
1,Brazil,United States,12,0.4286
2,South Korea,United States,15,0.6000


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Graph Overview**: The similarity heatmap shows how much the countries' top artist lists overlap. Darker cells mean the two countries share more artists, while lighter cells mean their lists are more different. Overall, the graph suggests that there is some shared global music taste across countries, but each country still has a distinct top artist profile.

**Interpretation:** 
- **South Korea and United States:** 15 shared artists, Jaccard similarity = **0.60**. This is the highest similarity, meaning these two countries have the most overlap in their top artist lists.

- **Brazil and United States:** 12 shared artists, Jaccard similarity = **0.4286**. This shows a moderate level of overlap between the two countries.

- **Brazil and South Korea:** 10 shared artists, Jaccard similarity = **0.3333**. This is the lowest similarity, meaning these two countries have the most different top artist lists.

**Data limitation:**
This data reflects Last.fm users who list their profile country as the United States, Brazil, or South Korea. It does not represent the entire population of those countries, since Last.fm users may have different music habits from people who do not use the platform. Because of this, the results should be interpreted as patterns among Last.fm users, not as a complete picture of national music preferences.

**Question 2:** *Which genres have the most listeners?*

**What the data measures:** Last.fm uses **tags** (loosely like genres), not a perfect genre tree. **“Listeners”** here means the **`reach`** field: how many users are associated with that tag ([`tag.getInfo`](https://www.last.fm/api/show/tag.getInfo)). We also show **`taggings`** (how often the tag was used). This section uses **`chart.getTopTags`** (global tag chart).

In [83]:
# Load the global genre/tag data from the JSON
tags_block = root["tags_df"]
tags_df = pd.DataFrame(tags_block["records"], columns=tags_block["columns"])

# Clean numeric columns
tags_df["reach"] = pd.to_numeric(tags_df["reach"], errors="coerce")
tags_df["taggings"] = pd.to_numeric(tags_df["taggings"], errors="coerce")

# Get the top genres/tags by global reach
top_genres = tags_df.sort_values("reach", ascending=False).head(15)

top_genres[["name", "reach", "taggings"]]

,name,reach,taggings
0,rock,403030,4070908
1,alternative,267280,2131501
2,electronic,262384,2501487
3,indie,260558,2065995
4,pop,233904,2084521
5,alternative rock,170338,1232172
6,female vocalists,169184,1634893
7,metal,159073,1306379
8,folk,151407,959834
9,ambient,150311,1127211


**Interpretation:**  
For this analysis, I used the `tags_df` data from the Last.fm dataset to compare global music genres/tags by reach. The `reach` column represents how widely each genre/tag appears across Last.fm users, so I used it as a proxy for global listener interest. I focused on the top genres by reach to see which music categories have the largest global audience on Last.fm.

The graph shows that broad genres such as rock, pop, and electronic have the highest global reach. This suggests that these genres are the most widely represented among Last.fm users. 

**Top 5 Genres by Global Reach:**
- **rock:** 403,030 reach
- **alternative:** 267,280 reach
- **electronic:** 262,384 reach
- **indie:** 260,558 reach
- **pop:** 233,904 reach

**Limitations:** However, because the data comes only from Last.fm accounts, the results should be interpreted as global patterns among Last.fm users, not as a complete measure of music listening across the entire world.

**Question 3:** Among the top 100 global artists, which 5 genres have the highest play-to-listener ratios?

The dataset is based on the Top 100 Global Artists, not all of the artists in the Last.fm database.

Per-artist enrichment is stored as **`rq3_artists_df`**; aggregated rankings as **`rq3_genre_summary`**.

In [63]:
# Load the genre summary data for play-to-listener ratio
genre_block = root["rq3_genre_summary"]
genre_summary = pd.DataFrame(
    genre_block["records"],
    columns=genre_block["columns"]
)

genre_summary

,genre_tag,median_play_per_listener,n_artists,sum_play,sum_listen,weighted_play_per_listener
0,rnb,138.946314,6,3.766162e+09,24967024.0,150.845459
1,Hip-Hop,122.964012,11,7.211050e+09,51393837.0,140.309627
2,pop,107.570439,23,1.067590e+10,101000173.0,105.701841
3,rock,103.920436,5,3.862198e+09,35156514.0,109.857245
4,indie,80.909842,6,2.802360e+09,22262589.0,125.877521


In [64]:
# Make sure numeric columns are treated as numbers
genre_summary["median_play_per_listener"] = pd.to_numeric(
    genre_summary["median_play_per_listener"],
    errors="coerce"
)

genre_summary["weighted_play_per_listener"] = pd.to_numeric(
    genre_summary["weighted_play_per_listener"],
    errors="coerce"
)

genre_summary["n_artists"] = pd.to_numeric(
    genre_summary["n_artists"],
    errors="coerce"
)

# Sort by median play-to-listener ratio
top_ratio_genres = genre_summary.sort_values(
    "median_play_per_listener",
    ascending=False
)

top_ratio_genres[[
    "genre_tag",
    "median_play_per_listener",
    "weighted_play_per_listener",
    "n_artists"
]]

,genre_tag,median_play_per_listener,weighted_play_per_listener,n_artists
0,rnb,138.946314,150.845459,6
1,Hip-Hop,122.964012,140.309627,11
2,pop,107.570439,105.701841,23
3,rock,103.920436,109.857245,5
4,indie,80.909842,125.877521,6


**Interpretation:**  

This graph shows the five genres with the highest median play-to-listener ratios among the top 100 global Last.fm artists. A higher ratio means that, on average, listeners are playing artists in that genre more times, which may suggest stronger repeat listening or fan engagement.

R&B has the highest play-to-listener ratio in this dataset, meaning its listeners appear to replay these artists more often than listeners of the other top genres shown. However, this result should be interpreted carefully because the analysis only includes top global artists, and each genre group may contain a small number of artists.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

**Research Question 1:** *Who are the top Last.fm artists by profile country (US, Brazil, South Korea) and how much do they overlap?*

**RQ1 Graph:** *Similarity of Top Artists by Country*

In [92]:
pairwise_block = root["pairwise_df"]
pairwise_df = pd.DataFrame(
    pairwise_block["records"],
    columns=pairwise_block["columns"]
)

countries = sorted(
    set(pairwise_df["profile_country_a"]).union(pairwise_df["profile_country_b"])
)

similarity_matrix = pd.DataFrame(1.0, index=countries, columns=countries)

for _, row in pairwise_df.iterrows():
    country_a = row["profile_country_a"]
    country_b = row["profile_country_b"]
    similarity = row["jaccard_similarity"]

    similarity_matrix.loc[country_a, country_b] = similarity
    similarity_matrix.loc[country_b, country_a] = similarity

fig = px.imshow(
    similarity_matrix,
    text_auto=".2f",
    color_continuous_scale="Blues",
    zmin=0,
    zmax=1,
    title="Overlap of Top Artists by Country",    
    labels={
        "color": "Jaccard Similarity"
    }
)
fig.update_layout(
    height=500,
    xaxis_title="Country",
    yaxis_title="Country",
    margin=dict(l=110, r=60, t=80, b=110)
)
fig.update_xaxes(
    title_standoff=25,
    ticklabelstandoff=15
)
fig.update_yaxes(
    title_standoff=35,
    ticklabelstandoff=15
)
fig.show()

**Chart rationale:**  
I chose a heatmap because the question is about similarity between pairs of countries, and a heatmap makes pairwise comparisons easy to scan. I want the reader to notice which country pairs have the most overlap in their top artist lists and which pairs have more distinct listening patterns.

I originally had a bar graph, but I didn't like how it visualized the data. It was kind of confusing, and I think a heatmap makes it easier for the audience to compare across two countries and check their similarity at once.

**Research Question 2:** *Which genres have the most listeners?*

**RQ2 Graph:** Top Global Genres by Reach

In [87]:
#I made a bar chart to show the top genres by reach

fig = px.bar(
    top_genres.sort_values("reach", ascending=True),
    x="reach",
    y="name",
    orientation="h",
    title="Rock Has the Highest Global Reach Among Last.fm Genres",
    labels={
        "reach": "Global Reach",
        "name": "Genre / Tag"
    },
    text="reach"
)

fig.update_traces(
    texttemplate="%{x:,.0f}",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    width=850,
    height=600,
    margin=dict(l=200, r=50, t=70, b=50)
)

fig.update_xaxes(
    tickformat=",",
    title_standoff=20,
    range=[0, 450000]
)

fig.update_yaxes(
    title_standoff=20,
    ticklabelstandoff=20
)

fig.show()

**Chart Rationale:**

I chose a horizontal bar chart because the genre names are easier to read horizontally, and the bars make it clear which genres have the largest global reach. I want the reader to understand which genres/tags are most popular among Last.fm users globally.


**Research Question 3:** *Among the top 100 global artists, which genres have the highest play-to-listener ratios?*

**RQ3 Graph:** Genres with the highest play-to-listener ratio

In [89]:
fig = px.bar(
    top_ratio_genres.sort_values("median_play_per_listener", ascending=True),
    x="median_play_per_listener",
    y="genre_tag",
    orientation="h",
    title="R&B Shows the Highest Play Intensity",
    labels={
        "median_play_per_listener": "Median Plays per Listener",
        "genre_tag": "Genre"
    },
    text="median_play_per_listener"
)

fig.update_traces(
    texttemplate="%{x:.1f}",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    width=800,
    height=450,
    margin=dict(l=150, r=120, t=70, b=50)
)

fig.update_xaxes(
    range=[0, 150]
)

fig.update_yaxes(
    title_standoff=20,
    ticklabelstandoff=20
)

fig.show()

**Chart Rationale**

I chose a horizontal bar chart because it clearly ranks the top genres by play-to-listener ratio and makes it easy to compare the strength of repeat listening across genres. I want the reader to take away that genres with higher ratios may have more engaged listeners, while also remembering that this is based only on the top global artist sample.


---

## Section 5 — Conclusions

**Summary of findings:** 

This analysis shows that Last.fm listening patterns have both global overlap and country-specific differences. The United States and South Korea had the most overlap in their top artist lists, while Brazil and South Korea were the least similar. I was surprised that R&B had the highest play-to-listener ratio among the top global artists, suggesting stronger play intensity in that genre. If I had more time, I would investigate more countries and more artists beyond the top 100. A limitation is that this data only represents Last.fm users, so it cannot describe all listeners in a country or the global music market as a whole.